In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import Window

# Read the bronze.crm.cust.info table

In [0]:
df = spark.table("bronze.crm_cust_info")

# Check the null data on primary key

In [0]:
df.select("*").filter(col("cst_id").isNull()).display()

So the null value in primary key caused by the data entry error, so we will just drop it
# Drop null value on primary key

In [0]:
df = df.dropna(subset=["cst_id"])

# **Check the duplicate data on primary key**

In [0]:
df.groupBy(
  col("cst_id")
).agg(count("cst_id").alias("total_id")).filter(col("total_id") > 1).display()

In [0]:
df.select("*").filter(col("cst_id").isin(29433,29449,29466,29473,29483)).display()

We learn that the duplicate data is hostocial data, so lets clean the old data
# **Cleaning historical data**

In [0]:
window = Window.partitionBy("cst_id").orderBy(col("cst_create_date").desc())

df = df.select("*",
          count("cst_create_date").over(window).alias("cst_id_index")
          ).filter(col("cst_id_index") == 1)
df = df.drop("cst_id_index")

# **Trimming all coloumn**

There are a lot of unwanted spaces

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

# Checking data standardization of cst_marital_status and cst_gndr

In [0]:
df.select('cst_marital_status').distinct().display()

In [0]:
df.select(
    col("cst_gndr")
).distinct().display()

# Standardize all the column

In [0]:
df = (df.
      withColumn("cst_marital_status", when(col("cst_marital_status") == "M", "Married")
                   .when(col("cst_marital_status") == "S", "Single")
                   .otherwise("N/A"))
      .withColumn("cst_gndr", when(col("cst_gndr") == "M", "Male")
                  .when(col("cst_gndr") == "F", "Female")
                  .otherwise("N/A"))
      )

# Sanity Check

In [0]:
df.limit(200).display()